# Event-level grouped LLM experiment

Цель: проверить гипотезу, что факторный каталог должен быть не статистическим
(`mortality_rate = коэффициент смертности`), а событийным
(`mortality_rate = гибель людей, погибшие, летальные исходы`), и что 36 факторов
лучше классифицировать группами.

В этом ноутбуке уже используются сохранённые результаты LLM-прогона:

- `balanced_recall_v2`: лучший одиночный baseline из прошлого ноутбука;
- `recall_max_v1_think_true`: high-recall prompt с thinking;
- `grouped_event_v1_search_think_true`: новый grouped event-level prompt.

Основная метрика остаётся `factor_micro_f1` по 36 бинарным факторным меткам.
Вспомогательные метрики объясняют, за счёт чего меняется качество.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, f1_score

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "analysis_outputs"
DATASET_PATH = OUTPUT_DIR / "interfax_news_multilabel_factor_dataset_1000_wide.csv"

from analyzer.factors import FACTOR_CONFIG
from analyzer.event_factor_taxonomy import EVENT_FACTOR_GROUPS, EVENT_FACTOR_TAXONOMY

FACTOR_KEYS = [item["key"] for item in FACTOR_CONFIG]
FACTOR_NAMES = {item["key"]: item["name"] for item in FACTOR_CONFIG}

paths = {
    "baseline_balanced": OUTPUT_DIR / "prompt_search_search_balanced_recall_v2_gemma4-e2b_5df03950348d_e98b1a3c5abc.csv",
    "thinking_recall": OUTPUT_DIR / "thinking_recall_experiment_recall_max_v1_think_true.csv",
    "grouped_event": OUTPUT_DIR / "grouped_event_v1_search_think_true.csv",
}
for name, path in paths.items():
    print(name, path.exists(), path)

baseline_balanced True C:\Users\whynot\VSCodeProjects\news-zero-shot\analysis_outputs\prompt_search_search_balanced_recall_v2_gemma4-e2b_5df03950348d_e98b1a3c5abc.csv
thinking_recall True C:\Users\whynot\VSCodeProjects\news-zero-shot\analysis_outputs\thinking_recall_experiment_recall_max_v1_think_true.csv
grouped_event True C:\Users\whynot\VSCodeProjects\news-zero-shot\analysis_outputs\grouped_event_v1_search_think_true.csv


## Taxonomy sanity check

Проверяем, что event-level каталог покрывает все 36 факторов ровно один раз
в grouped pipeline.

In [2]:
covered = [key for group in EVENT_FACTOR_GROUPS.values() for key in group["factor_keys"]]
taxonomy_check = {
    "factor_count": len(FACTOR_KEYS),
    "taxonomy_entries": len(EVENT_FACTOR_TAXONOMY),
    "grouped_entries": len(covered),
    "missing_from_groups": sorted(set(FACTOR_KEYS) - set(covered)),
    "duplicate_in_groups": sorted({key for key in covered if covered.count(key) > 1}),
    "missing_from_event_taxonomy": sorted(set(FACTOR_KEYS) - set(EVENT_FACTOR_TAXONOMY)),
}
taxonomy_check

{'factor_count': 36,
 'taxonomy_entries': 36,
 'grouped_entries': 36,
 'missing_from_groups': [],
 'duplicate_in_groups': [],
 'missing_from_event_taxonomy': []}

In [3]:
group_rows = []
for group_id, payload in EVENT_FACTOR_GROUPS.items():
    for key in payload["factor_keys"]:
        event = EVENT_FACTOR_TAXONOMY[key]
        group_rows.append({
            "group": group_id,
            "factor_key": key,
            "dashboard_name": FACTOR_NAMES[key],
            "event_name": event["event_name"],
            "event_definition": event["include"],
        })
pd.DataFrame(group_rows)

,group,factor_key,dashboard_name,event_name,event_definition
0,safety_mortality,crime_count,Количество преступлений,"Преступность, насилие и угрозы безопасности","преступления, уголовные дела, атаки, обстрелы,..."
1,safety_mortality,mortality_rate,Коэффициент смертности на 1000 человек,Гибель людей и смертность как социальный сигнал,"гибель людей, число погибших, смерть, летальны..."
2,safety_mortality,life_expectancy,Ожидаемая продолжительность жизни,Продолжительность и безопасность жизни,"угрозы жизни, тяжелые травмы, массовые заболев..."
3,safety_mortality,infant_mortality,Коэффициент младенческой смертности,Младенческая смертность и смерть младенцев,"смерть младенцев, гибель новорожденных, младен..."
4,healthcare,hospitals,Число больничных организаций,"Больницы, скорая помощь и стационарная медицина","больницы, госпитализация, скорая помощь, стаци..."
5,healthcare,outpatient_clinics,Число амбулаторно-поликлинических организаций,Поликлиники и первичная медпомощь,"поликлиники, амбулаторная помощь, первичное зв..."
6,healthcare,qualified_doctors,Численность врачей высшей и первой квалификаци...,Врачи и медицинские кадры,"врачи, медицинский персонал, дефицит кадров, к..."
7,healthcare,abortions,Число абортов,Аборты и репродуктивные решения,"аборты, ограничения/доступность абортов, репро..."
8,economy_income_prices,consumer_price_index,Индексы потребительских цен,Потребительские цены и тарифы,"инфляция, потребительские цены, тарифы ЖКХ, ст..."
9,economy_income_prices,per_capita_income,Среднедушевые денежные доходы населения,Денежные доходы населения,"зарплаты, пенсии, доходы, выплаты, покупательн..."


## Metric functions

`sample_f1_empty_correct` считается так же, как в предыдущем multilabel notebook:
если в новости нет gold-факторов и модель тоже ничего не вернула, это 1.0.

In [4]:
df = pd.read_csv(DATASET_PATH, encoding="utf-8-sig")
df["dataset_row_id"] = pd.to_numeric(df[df.columns[0]], errors="raise").astype(int)

y_true_df = pd.DataFrame(
    {
        key: pd.to_numeric(df[f"factor__{key}"], errors="coerce").fillna(0).astype(int).clip(0, 1).to_numpy()
        for key in FACTOR_KEYS
    },
    index=df["dataset_row_id"],
)

def load_scores(path: Path) -> pd.DataFrame:
    pred = pd.read_csv(path)
    ids = sorted(pred["dataset_row_id"].unique().astype(int).tolist())
    scores = pd.DataFrame(0.0, index=ids, columns=FACTOR_KEYS)
    for key, sub in pred.groupby("factor_key"):
        if key in scores.columns:
            values = sub.groupby("dataset_row_id")["relevance"].max()
            scores.loc[values.index.astype(int), key] = values.astype(float).values
    return scores

scores = {name: load_scores(path) for name, path in paths.items()}
ids = sorted(set.intersection(*(set(frame.index) for frame in scores.values())))
Y = y_true_df.loc[ids, FACTOR_KEYS].to_numpy(int)

def sample_f1_empty_correct(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    values = []
    for true_row, pred_row in zip(y_true, y_pred):
        true_sum = int(true_row.sum())
        pred_sum = int(pred_row.sum())
        tp = int(((true_row == 1) & (pred_row == 1)).sum())
        if true_sum == 0 and pred_sum == 0:
            values.append(1.0)
        elif true_sum == 0 or pred_sum == 0:
            values.append(0.0)
        else:
            values.append(2 * tp / (true_sum + pred_sum))
    return float(np.mean(values))

def evaluate_matrix(y_pred: np.ndarray, name: str) -> dict:
    micro = precision_recall_fscore_support(Y, y_pred, average="micro", zero_division=0)
    supported = Y.sum(axis=0) > 0
    any_metric = precision_recall_fscore_support(
        Y.sum(axis=1) > 0,
        y_pred.sum(axis=1) > 0,
        average="binary",
        zero_division=0,
    )
    return {
        "name": name,
        "pred_positive_pairs": int(y_pred.sum()),
        "mean_labels_per_news": float(y_pred.sum(axis=1).mean()),
        "factor_micro_precision": float(micro[0]),
        "factor_micro_recall": float(micro[1]),
        "factor_micro_f1": float(micro[2]),
        "factor_macro_f1_supported": float(f1_score(Y[:, supported], y_pred[:, supported], average="macro", zero_division=0)),
        "sample_f1_empty_correct": sample_f1_empty_correct(Y, y_pred),
        "any_relevant_precision": float(any_metric[0]),
        "any_relevant_recall": float(any_metric[1]),
        "any_relevant_f1": float(any_metric[2]),
        "false_relevant_news": int(((Y.sum(axis=1) == 0) & (y_pred.sum(axis=1) > 0)).sum()),
        "missed_all_relevant_news": int(((Y.sum(axis=1) > 0) & (y_pred.sum(axis=1) == 0)).sum()),
    }

print("search ids:", len(ids))
print("gold positive factor pairs:", int(Y.sum()))

search ids: 300
gold positive factor pairs: 307


## Main scoreboard

Сравниваем одиночные модели, старый ensemble и новый event-level grouped ensemble.

In [5]:
B = scores["baseline_balanced"].loc[ids, FACTOR_KEYS].to_numpy(float)
T = scores["thinking_recall"].loc[ids, FACTOR_KEYS].to_numpy(float)
G = scores["grouped_event"].loc[ids, FACTOR_KEYS].to_numpy(float)

variants = {
    "balanced_recall_v2@0.30": (B >= 0.30).astype(int),
    "thinking_recall@0.30": (T >= 0.30).astype(int),
    "grouped_event@0.35": (G >= 0.35).astype(int),
    "grouped_event@0.05_high_recall": (G >= 0.05).astype(int),
    "old_ensemble_baseline+thinking@0.35": np.logical_or(B >= 0.35, T >= 0.35).astype(int),
    "new_ensemble_baseline+thinking+grouped@0.35": np.logical_or.reduce([B >= 0.35, T >= 0.35, G >= 0.35]).astype(int),
    "new_ensemble_high_recall@0.10/0.35/0.35": np.logical_or.reduce([B >= 0.10, T >= 0.35, G >= 0.35]).astype(int),
}

scoreboard = pd.DataFrame([evaluate_matrix(pred, name) for name, pred in variants.items()])
scoreboard.sort_values(["factor_micro_f1", "sample_f1_empty_correct"], ascending=False).round(4)

,name,pred_positive_pairs,mean_labels_per_news,factor_micro_precision,factor_micro_recall,factor_micro_f1,factor_macro_f1_supported,sample_f1_empty_correct,any_relevant_precision,any_relevant_recall,any_relevant_f1,false_relevant_news,missed_all_relevant_news
5,new_ensemble_baseline+thinking+grouped@0.35,295,0.9833,0.5492,0.5277,0.5382,0.3633,0.5632,0.9105,0.7621,0.8297,17,54
4,old_ensemble_baseline+thinking@0.35,208,0.6933,0.6635,0.4495,0.5359,0.2759,0.5629,0.9261,0.7181,0.8089,13,64
6,new_ensemble_high_recall@0.10/0.35/0.35,372,1.2400,0.4785,0.5798,0.5243,0.3508,0.5415,0.8589,0.9119,0.8846,34,20
0,balanced_recall_v2@0.30,236,0.7867,0.5593,0.4300,0.4862,0.2469,0.5119,0.8732,0.8194,0.8455,27,41
1,thinking_recall@0.30,253,0.8433,0.5296,0.4365,0.4786,0.1798,0.4859,0.8292,0.8767,0.8522,41,28
2,grouped_event@0.35,165,0.5500,0.4848,0.2606,0.3390,0.2778,0.4071,0.9423,0.4317,0.5921,6,129
3,grouped_event@0.05_high_recall,636,2.1200,0.2469,0.5114,0.3330,0.2675,0.3570,0.8091,0.8590,0.8333,46,32


## Threshold and tuning result

`grouped_event` сам по себе оказался слишком слабым по micro-F1. Его ценность не в одиночном режиме,
а в том, что он добавляет часть редких/спорных факторов к ensemble.

In [6]:
grouped_metrics = pd.read_csv(OUTPUT_DIR / "grouped_event_v1_search_think_true_metrics.csv")
grouped_metrics[
    [
        "mode",
        "threshold",
        "factor_micro_f1",
        "factor_micro_precision",
        "factor_micro_recall",
        "factor_macro_f1_supported",
        "sample_f1_empty_correct",
        "any_relevant_f1",
        "mean_pred_labels_per_news",
        "false_relevant_news",
        "missed_all_relevant_news",
    ]
].head(12).round(4)

,mode,threshold,factor_micro_f1,factor_micro_precision,factor_micro_recall,factor_macro_f1_supported,sample_f1_empty_correct,any_relevant_f1,mean_pred_labels_per_news,false_relevant_news,missed_all_relevant_news
0,threshold,0.35,0.339,0.4848,0.2606,0.2778,0.4071,0.5921,0.55,6,129
1,threshold,0.40,0.339,0.4848,0.2606,0.2778,0.4071,0.5921,0.55,6,129
2,threshold,0.45,0.339,0.4848,0.2606,0.2778,0.4071,0.5921,0.55,6,129
3,threshold,0.50,0.339,0.4848,0.2606,0.2778,0.4071,0.5921,0.55,6,129
4,threshold,0.55,0.339,0.4848,0.2606,0.2778,0.4071,0.5921,0.55,6,129
5,threshold,0.60,0.339,0.4848,0.2606,0.2778,0.4071,0.5921,0.55,6,129
6,threshold,0.05,0.333,0.2469,0.5114,0.2675,0.3570,0.8333,2.12,46,32
7,threshold,0.10,0.333,0.2469,0.5114,0.2675,0.3570,0.8333,2.12,46,32
8,threshold,0.15,0.333,0.2469,0.5114,0.2675,0.3570,0.8333,2.12,46,32
9,threshold,0.20,0.333,0.2469,0.5114,0.2675,0.3570,0.8333,2.12,46,32


In [7]:
hybrid_metrics = pd.read_csv(OUTPUT_DIR / "grouped_event_hybrid_search_metrics_empty_correct.csv")
hybrid_metrics.head(12).round(4)

,name,pairs,precision,recall,f1,macro_supported,sample_f1_empty_correct,any_f1,false_relevant,missed_all
0,union:baseline_balanced@0.35+thinking_recall@0...,295,0.5492,0.5277,0.5382,0.3633,0.5632,0.8297,17,54
1,union:baseline_balanced@0.35+thinking_recall@0...,295,0.5492,0.5277,0.5382,0.3633,0.5632,0.8297,17,54
2,union:baseline_balanced@0.35+thinking_recall@0...,295,0.5492,0.5277,0.5382,0.3633,0.5632,0.8297,17,54
3,union:baseline_balanced@0.35+thinking_recall@0...,295,0.5492,0.5277,0.5382,0.3633,0.5632,0.8297,17,54
4,union:baseline_balanced@0.6+thinking_recall@0....,295,0.5492,0.5277,0.5382,0.3633,0.5632,0.8297,17,54
5,union:baseline_balanced@0.6+thinking_recall@0....,295,0.5492,0.5277,0.5382,0.3633,0.5632,0.8297,17,54
6,union:baseline_balanced@0.6+thinking_recall@0....,295,0.5492,0.5277,0.5382,0.3633,0.5632,0.8297,17,54
7,union:baseline_balanced@0.6+thinking_recall@0....,295,0.5492,0.5277,0.5382,0.3633,0.5632,0.8297,17,54
8,union:baseline_balanced@0.35+thinking_recall@0.35,208,0.6635,0.4495,0.5359,0.2759,0.5629,0.8089,13,64
9,union:baseline_balanced@0.35+thinking_recall@0.6,208,0.6635,0.4495,0.5359,0.2759,0.5629,0.8089,13,64


In [8]:
holdout_metrics = pd.read_csv(OUTPUT_DIR / "grouped_event_hybrid_factor_tuned_holdout_metrics.csv")
holdout_metrics.round(4)

,name,pairs,precision,recall,f1,macro_supported,sample_f1_empty_correct,any_f1,false_rel,missed_all
0,factor_tuned_on_tune/eval=tune,132,0.6136,0.5586,0.5848,0.4451,0.6062,0.8235,13,23
1,factor_tuned_on_tune/eval=heldout,126,0.6270,0.4877,0.5486,0.2734,0.5544,0.8349,7,29
2,factor_tuned_on_tune/eval=all_search,258,0.6202,0.5212,0.5664,0.3216,0.5803,0.8294,20,52
3,uniform_union_0.35/eval=heldout,159,0.5786,0.5679,0.5732,0.3961,0.5717,0.8479,5,28
4,uniform_union_0.35/eval=all_search,295,0.5492,0.5277,0.5382,0.3633,0.5632,0.8297,17,54


## Per-factor diagnostics

Смотрим, где новый grouped prompt добавил пользу, а где испортил качество.

In [9]:
def per_factor_metrics(y_pred: np.ndarray, label: str) -> pd.DataFrame:
    rows = []
    for i, key in enumerate(FACTOR_KEYS):
        pr = precision_recall_fscore_support(Y[:, i], y_pred[:, i], average="binary", zero_division=0)
        rows.append({
            "factor_key": key,
            "factor_name": FACTOR_NAMES[key],
            "support": int(Y[:, i].sum()),
            "predicted_positive": int(y_pred[:, i].sum()),
            "precision": float(pr[0]),
            "recall": float(pr[1]),
            "f1": float(pr[2]),
            "variant": label,
        })
    return pd.DataFrame(rows)

old_ensemble = variants["old_ensemble_baseline+thinking@0.35"]
new_ensemble = variants["new_ensemble_baseline+thinking+grouped@0.35"]
grouped_only = variants["grouped_event@0.35"]

per_old = per_factor_metrics(old_ensemble, "old_ensemble")
per_new = per_factor_metrics(new_ensemble, "new_ensemble")
per_grouped = per_factor_metrics(grouped_only, "grouped_only")

factor_compare = (
    per_new[["factor_key", "factor_name", "support", "predicted_positive", "precision", "recall", "f1"]]
    .merge(
        per_old[["factor_key", "predicted_positive", "precision", "recall", "f1"]],
        on="factor_key",
        suffixes=("_new", "_old"),
    )
)
factor_compare["f1_delta_vs_old"] = factor_compare["f1_new"] - factor_compare["f1_old"]
factor_compare["recall_delta_vs_old"] = factor_compare["recall_new"] - factor_compare["recall_old"]
factor_compare[factor_compare["support"] > 0].sort_values("f1_delta_vs_old", ascending=False).round(4)

,factor_key,factor_name,support,predicted_positive_new,precision_new,recall_new,f1_new,predicted_positive_old,precision_old,recall_old,f1_old,f1_delta_vs_old,recall_delta_vs_old
27,primary_housing_price_index,Индексы цен на первичном рынке жилья,2,1,1.0000,0.5000,0.6667,0,0.0000,0.0000,0.0000,0.6667,0.5000
0,population_size,Численность населения,1,3,0.3333,1.0000,0.5000,0,0.0000,0.0000,0.0000,0.5000,1.0000
20,preschool_coverage,Охват детей до 6 лет дошкольным образованием,3,1,1.0000,0.3333,0.5000,0,0.0000,0.0000,0.0000,0.5000,0.3333
25,housing_area_per_capita,Общая площадь жилья на одного жителя,9,7,0.2857,0.2222,0.2500,0,0.0000,0.0000,0.0000,0.2500,0.2222
11,per_capita_income,Среднедушевые денежные доходы населения,10,8,0.3750,0.3000,0.3333,1,1.0000,0.1000,0.1818,0.1515,0.2000
34,mortality_rate,Коэффициент смертности на 1000 человек,13,15,0.6000,0.6923,0.6429,6,0.8333,0.3846,0.5263,0.1165,0.3077
33,wastewater_discharge,Сброс загрязненных сточных вод,4,5,0.6000,0.7500,0.6667,3,0.6667,0.5000,0.5714,0.0952,0.2500
26,consumer_price_index,Индексы потребительских цен,22,26,0.5385,0.6364,0.5833,14,0.6429,0.4091,0.5000,0.0833,0.2273
30,enterprises_count,Число предприятий и организаций,44,26,0.7308,0.4318,0.5429,23,0.7391,0.3864,0.5075,0.0354,0.0455
31,crime_count,Количество преступлений,70,64,0.7656,0.7000,0.7313,60,0.7667,0.6571,0.7077,0.0237,0.0429


In [10]:
grouped_report = pd.read_csv(OUTPUT_DIR / "grouped_event_v1_search_think_true_factor_report.csv")
grouped_report[grouped_report["support"] > 0].sort_values("f1", ascending=False).round(4)

,factor_key,support,predicted_positive,precision,recall,f1
19,primary_housing_price_index,2,1,1.0000,0.5000,0.6667
16,living_wage,2,1,1.0000,0.5000,0.6667
5,mortality_rate,13,14,0.6429,0.6923,0.6667
3,consumer_price_index,22,19,0.5789,0.5000,0.5366
22,population_size,1,3,0.3333,1.0000,0.5000
15,preschool_coverage,3,1,1.0000,0.3333,0.5000
6,international_inflow,12,12,0.4167,0.4167,0.4167
0,crime_count,70,23,0.8261,0.2714,0.4086
1,industrial_production_index,64,23,0.6522,0.2344,0.3448
18,children_benefits,2,4,0.2500,0.5000,0.3333


## Conclusion

1. Event-level taxonomy как идея правильная: `mortality_rate`, `hospitals`, часть housing/children факторов
стали лучше объяснимы, а `factor_macro_f1_supported` в новом ensemble вырос.

2. Grouped-only pipeline не надо выкатывать как замену baseline: лучший grouped-only micro-F1 на search
около 0.339, это хуже текущего balanced baseline.

3. Лучший практический кандидат из уже прогнанных вариантов: union ensemble
`balanced_recall_v2 + recall_max_thinking + grouped_event`, threshold около 0.35.
Он даёт лучший общий баланс на search: выше recall и macro-supported, но precision проседает.

4. Ожидание F1 0.7-0.8 одним prompt-тюнингом здесь не подтверждается. Следующий правильный шаг:
двухстадийная схема candidate generator -> verifier/reranker по парам news-factor, желательно с
supervised calibration на этом gold dataset.